# Adaptive Threshold Evaluation on GSM8K

测试自适应阈值公式 $\tau^{(t)} = \tau_0 (1 - \alpha(1 - r_{\text{mask}}))$ 在 GSM8K 上的效果。

**参数：**
- $\tau_0 = 0.9$ (base threshold)
- $\alpha = 0.3$ (adaptation strength)
- $r_{\text{mask}}$ = 当前 block 的 mask 比例

**两组实验（2 个 GPU 并行）：**
1. **dual_cache** — 标准 dual cache 模式 + adaptive threshold
2. **expand_rewarm_fb_on** — Dual Cache + Mid-Block Expansion (rewarm=True, fb=True) + adaptive threshold

**对比基线（来自之前实验）：**
- expand_rewarm_fb_on (fixed τ=0.9): flexible-extract = 0.7945, NFE = 99087

## 1. 环境设置

In [ ]:
import os
import torch
import gc

os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3,4,5,6,7'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

os.chdir('llada')
os.makedirs('nlogs', exist_ok=True)

torch.cuda.empty_cache()
gc.collect()

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. 启动两组实验

两组实验在不同 GPU 上并行运行全量 GSM8K (1319 题)。

核心改动：`adaptive_alpha=0.3`，使得阈值随 mask 比例动态调整。

In [ ]:
import subprocess
import datetime

task = "gsm8k"
fewshot = 5
limit = None  # 全量
seed = 42
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# ========== 通用生成参数 ==========
gen_length = 256
steps = 256
block_length = 32
threshold = 0.9
adaptive_alpha = 0.3

# ========== 两组配置 ==========
configs = [
    (0, "dualcache_adaptive", [
        "dual_cache=True",
        f"adaptive_alpha={adaptive_alpha}",
    ]),
    (1, "expand_rewarm_fb_on_adaptive", [
        "dual_cache=True",
        "mid_block_expand=True",
        "mid_trigger_ratio=0.5",
        "rewarm_on_expand=True",
        "front_block_fallback_only=True",
        f"adaptive_alpha={adaptive_alpha}",
    ]),
]

processes = []

for gpu, name, extra_args in configs:
    log_file = f"nlogs/adaptive_{task}_{name}_{timestamp}.log"
    output_dir = f"evals_results/adaptive/{task}-{name}-{timestamp}"
    records_dir = f"evals_results/adaptive/{task}-{name}-{timestamp}/step_records"

    base_args = [
        f"model_path='GSAI-ML/LLaDA-8B-Instruct'",
        f"gen_length={gen_length}",
        f"steps={steps}",
        f"block_length={block_length}",
        f"threshold={threshold}",
        "use_cache=True",
        "show_speed=True",
        f"step_records_dir='{records_dir}'",
        f"seed={seed}",
    ]
    all_args = base_args + extra_args
    model_args_str = ",".join(all_args)

    limit_arg = f"--limit {limit}" if limit is not None else ""

    cmd = (
        f"CUDA_VISIBLE_DEVICES={gpu} accelerate launch eval_llada.py "
        f"--tasks {task} --num_fewshot {fewshot} {limit_arg} "
        f"--confirm_run_unsafe_code --model llada_dist "
        f"--model_args {model_args_str} "
        f"--output_path {output_dir} --log_samples"
    )

    print(f"GPU {gpu} | {name}")
    print(f"  τ_0={threshold}, α={adaptive_alpha}")
    print(f"  Log: {log_file}")
    print(f"  Cmd: {cmd[:200]}...")
    print()

    p = subprocess.Popen(cmd, shell=True, stdout=open(log_file, "w"), stderr=subprocess.STDOUT)
    processes.append((p, name, log_file, output_dir))

print(f"\n{len(processes)} tasks launched in parallel, waiting...")

In [ ]:
# 等待所有进程完成
for p, name, log, out_dir in processes:
    p.wait()
    rc = p.returncode
    status = "OK" if rc == 0 else f"FAILED (exit {rc})"
    print(f"{name}: {status}")

    with open(log, 'r') as f:
        lines = f.readlines()
    print(f"  Last lines:")
    for line in lines[-15:]:
        print(f"    {line.rstrip()}")
    print()

## 3. 解析评测结果

In [ ]:
import glob
import re
import json
import numpy as np
import pandas as pd

log_files = sorted(glob.glob(f"nlogs/adaptive_{task}_*_{timestamp}.log"))

print(f"Adaptive Threshold GSM8K Results (τ_0={threshold}, α={adaptive_alpha}, timestamp={timestamp}):")
print("=" * 100)

parsed_results = []
for log_file in log_files:
    fname = os.path.basename(log_file)
    name_part = fname.replace(f"adaptive_{task}_", "").replace(f"_{timestamp}.log", "")

    with open(log_file, 'r') as f:
        content = f.read()

    # Extract flexible-extract accuracy
    flex_match = re.search(r'flexible-extract.*?exact_match.*?([\d.]+)', content)
    flex_acc = float(flex_match.group(1)) if flex_match else None

    # Extract strict-match accuracy
    strict_match = re.search(r'strict-match.*?exact_match.*?([\d.]+)', content)
    strict_acc = float(strict_match.group(1)) if strict_match else None

    speed_match = re.search(r'Tokens per second:\s*([\d.]+)', content)
    speed = float(speed_match.group(1)) if speed_match else None

    nfe_match = re.search(r'Total NFE is (\d+)', content)
    nfe = int(nfe_match.group(1)) if nfe_match else None

    time_match = re.search(r'Total time taken:\s*([\d.]+)', content)
    time_sec = float(time_match.group(1)) if time_match else None

    tokens_match = re.search(r'Total number of tokens generated:\s*(\d+)', content)
    total_tokens = int(tokens_match.group(1)) if tokens_match else None

    parsed_results.append({
        'config': name_part,
        'flex_acc': flex_acc,
        'strict_acc': strict_acc,
        'tokens_per_sec': speed,
        'total_nfe': nfe,
        'total_tokens': total_tokens,
        'time_sec': time_sec,
        'log_file': log_file,
    })

# 当前结果
print(f"{'Config':<35} {'Flex Acc':<10} {'Strict Acc':<12} {'Tok/s':<10} {'NFE':<10} {'Time(s)':<10}")
print("-" * 90)
for r in parsed_results:
    flex_str = f"{r['flex_acc']:.4f}" if r['flex_acc'] is not None else "N/A"
    strict_str = f"{r['strict_acc']:.4f}" if r['strict_acc'] is not None else "N/A"
    speed_str = f"{r['tokens_per_sec']:.1f}" if r['tokens_per_sec'] is not None else "N/A"
    nfe_str = str(r['total_nfe']) if r['total_nfe'] is not None else "N/A"
    time_str = f"{r['time_sec']:.1f}" if r['time_sec'] is not None else "N/A"
    print(f"{r['config']:<35} {flex_str:<10} {strict_str:<12} {speed_str:<10} {nfe_str:<10} {time_str:<10}")

# 对比基线
print("\n" + "=" * 90)
print("基线对比 (fixed τ=0.9, expand_rewarm_fb_on):")
print(f"  Flex Acc: 0.7945 | NFE: 99087 | Tok/s: 31.2 | Time: 9826.1s")

df = pd.DataFrame(parsed_results)
display(df)

## 4. 公式效果说明

自适应阈值公式 $\tau^{(t)} = \tau_0 (1 - \alpha(1 - r_{\text{mask}}))$

| $r_{\text{mask}}$ | $\tau^{(t)}$ (α=0.3) | 含义 |
|---|---|---|
| 1.0 (全 mask) | 0.90 | 初始步骤，阈值最严格 |
| 0.5 (半解码) | 0.765 | 中间步骤，阈值放松 |
| 0.0 (全解码) | 0.63 | 最后步骤，阈值最宽松 |

**预期效果：** 解码后期 mask 减少时阈值降低，允许更多 token 通过，减少 NFE，提升推理速度。
关键是观察准确率是否能维持在基线水平附近。